In [4]:
import requests, json
from bs4 import BeautifulSoup

URL = r"https://www.athleticsireland.ie/competitions-events/fixtures-results/"
HEADERS = {"User-Agent": "RaceTrace student project"}

In [6]:
title = "Track & Field Live #3"

html = requests.get(URL, headers=HEADERS).text
soup = BeautifulSoup(html, "html.parser")

hit = soup.find(string=lambda s: s and title in s)
node = hit.parent
for _ in range(4):
    print("-----", node.name, node.get("class"))
    node = node.parent

print(hit.parent.parent.prettify()[:2000])

----- div ['text-brand-dark-grey', 'font-semibold', 'leading-6']
----- div ['grid', 'items-center', 'gap-y-2', 'lg:gap-y-0', 'lg:grid-cols-[140px_minmax(200px,1fr)]']
----- div ['fixture-card', 'w-full', 'grid', 'lg:grid-cols-[1fr_minmax(150px,300px)_1fr]', 'gap-x-[30px]']
----- div ['outline', 'outline-1', 'outline-white', 'transition-all', 'header-scroll-margin', 'fixture-card-wrapper', 'flex', 'rounded-lg', 'p-[30px]', 'bg-brand-off-white']
<div class="grid items-center gap-y-2 lg:gap-y-0 lg:grid-cols-[140px_minmax(200px,1fr)]">
 <div class="flex justify-between items-center lg:flex-none text-brand-mid-grey text-sm">
  <div class="flex before:icon--calendar text-brand-dark-grey">
   <span class="pl-1 text-brand-mid-grey">
    14 Jan ‘26
   </span>
  </div>
  <div class="block lg:hidden">
   <div class="">
    <span class="pill pill--fixture inline-block font-medium text-xs rounded-[13px] px-2.5 py-1.25">
     Nat _P
    </span>
   </div>
  </div>
 </div>
 <div class="text-brand-dark

In [9]:
card = soup.select_one("div.fixture-card")
print(card.parent.attrs)

{'data-fixture-id': '54488', 'class': ['outline', 'outline-1', 'outline-white', 'transition-all', 'header-scroll-margin', 'fixture-card-wrapper', 'flex', 'rounded-lg', 'p-[30px]', 'bg-brand-off-white']}


In [17]:
def text(card, selector):
    el = card.select_one(selector)
    return el.get_text(strip=True) if el else ""

races, bad = [], []
for card in soup.select("div.fixture-card"):
    title_el = card.select_one(".font-semibold")
    row = {
        "id": card.parent.get("data-fixture-id", ""),
        "date": text(card, "div[class*='icon--calendar'] span"),
        "title": title_el.get_text(strip=True) if title_el else "",
        "url": title_el.get("href", "") if title_el else "",
        "location": text(card, "div[class*='icon--map-pin'] span"),
        "leve": text(card, "span.pill")
    }
    (races if row["title"] else bad).append(row if row["title"] else card)

print(len(races), "ok,", len(bad), "bad")

182 ok, 0 bad


In [28]:
import pandas as pd

df = pd.DataFrame(races)
df.head(20)


,id,date,title,url,location,leve
0,54488,10 Jan ‘26,World Cross Country,,"Tallahassee, US",International
1,52256,14 Jan ‘26,Track & Field Live #3,,"Dublin 15, IE",Nat _P
2,53436,17-18 Jan ‘26,National Indoor AAI Games & Combines SYJ Day 1,https://www.athleticsireland.ie/competitions-e...,"Dublin 15, IE",National
3,53435,18 Jan ‘26,"National Indoor CE SYJ Day 2 & CE Juvenile, Ma...",,"Dublin 15, IE",National
4,53660,23 Jan ‘26,North Leinster Schools Cross Country Champions...,,"Dublin 8, IE",Schools
5,53437,24 Jan ‘26,National Indoor League Rd 1 & Guest Open,,"Dublin 15, IE",National
6,53782,25 Jan ‘26,Dublin Indoor Juvenile Championships Day 1,,"Dublin 15, IE",Regional
7,53661,27 Jan ‘26,South Leinster Schools Cross Country Champions...,,"Carlow, IE",Schools
8,52258,28 Jan ‘26,Track & Field Live #5,,"Dublin 15, IE",Nat _P
9,53783,31 Jan ‘26,Dublin Indoor Juvenile Championships Day 2,,"Dublin 15, IE",Regional


In [29]:
import re
from datetime import date

MONTHS = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
          "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}

def parse_date(text):
    day = int(re.search(r"\d{1,2}", text).group())
    month = MONTHS[re.search(r"[A-Za-z]{3}", text).group().lower()]
    year = 2000 + int(re.search(r"(\d{2})\s*$", text).group(1))
    return date(year, month, day)

parse_date("17-18 jan '26")

datetime.date(2026, 1, 17)

In [31]:
df["date"] = df["date"].apply(parse_date)

df.head()

,id,date,title,url,location,leve
0,54488,2026-01-10,World Cross Country,,"Tallahassee, US",International
1,52256,2026-01-14,Track & Field Live #3,,"Dublin 15, IE",Nat _P
2,53436,2026-01-17,National Indoor AAI Games & Combines SYJ Day 1,https://www.athleticsireland.ie/competitions-e...,"Dublin 15, IE",National
3,53435,2026-01-18,"National Indoor CE SYJ Day 2 & CE Juvenile, Ma...",,"Dublin 15, IE",National
4,53660,2026-01-23,North Leinster Schools Cross Country Champions...,,"Dublin 8, IE",Schools
